# Multi-Agent Workflows
Environment setup and tool definitions for the multi-agent module.

In [2]:
import os
import requests
from typing import TypedDict, List
from dotenv import load_dotenv
from pydantic import BaseModel
from agents import Agent, Runner, function_tool, SQLiteSession

In [3]:
load_dotenv()

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")

assert OPENAI_API_KEY, "OPENAI_API_KEY not set in .env"
assert TAVILY_API_KEY, "TAVILY_API_KEY not set in .env"

print("API keys loaded.")

API keys loaded.


In [4]:
class TavilySearchInput(TypedDict):
    query: str
    max_results: int


def _tavily_search_impl(query: str, max_results: int = 2) -> str:
    payload = {
        "api_key": TAVILY_API_KEY,
        "query": query,
        "max_results": max_results,
    }
    response = requests.post("https://api.tavily.com/search", json=payload)
    response.raise_for_status()
    results = response.json().get("results", [])
    if not results:
        return "No results found."
    return "\n\n".join(
        f"Title: {r['title']}\nURL: {r['url']}\nContent: {r['content']}"
        for r in results
    )


@function_tool
def tavily_search(search_params: TavilySearchInput) -> str:
    """Search the web using the Tavily API and return formatted results."""
    return _tavily_search_impl(
        search_params["query"],
        search_params.get("max_results", 2),
    )

In [ ]:
# Smoke-test — calls the HTTP layer directly, no SDK context needed
result = _tavily_search_impl("OpenAI Agents SDK", max_results=2)
print(result)

## LinkedIn Agents

Three specialized agents tuned for professional social content. The **Researcher** pulls real-time facts via Tavily. The **Analyst** extracts career, business, and industry implications for a LinkedIn audience. The **Writer** produces a polished post or article in the chosen format — with emojis, references, and hashtags.

In [5]:
class ResearchOutput(BaseModel):
    summary: str

In [6]:
researcher = Agent(
    name="Researcher",
    model="gpt-5-mini",
    instructions=(
        "You are a research assistant. Use the tavily_search tool to find real-time information "
        "on the given topic. Return your findings as exactly 5 concise bullet points, each "
        "capturing a distinct fact or insight. Format your summary as a bulleted list."
    ),
    tools=[tavily_search],
    output_type=ResearchOutput,
)

analyst = Agent(
    name="Analyst",
    model="gpt-5-mini",
    instructions=(
        "You are an expert analyst writing for a LinkedIn professional audience. "
        "You receive research notes and produce exactly 2 paragraphs:\n"
        "1. Career and skill implications: what practitioners need to know, learn, or do.\n"
        "2. Business and industry impact: how this trend is moving the market and what leaders should watch."
    ),
    output_type=ResearchOutput,
)

In [ ]:
query = "Why is Labubu so popular and what are the rarest Labubu collectibles?"

research_result = await Runner.run(researcher, input=query)
print(research_result.final_output.summary)

## LinkedIn Pipeline

`manager_run(user_query, format_type)` chains all three agents over a shared `SQLiteSession`. Set `format_type` in the cell below before running.

| `format_type` | Best for |
|---|---|
| `"short_post"` | Quick insight, 150–300 words, bullets + CTA |
| `"long_article"` | Thought leadership, 700–1200 words, referenced |
| `"celebration"` | Milestone / achievement, warm + personal |
| `"reaction"` | Response to a trend or news item, clear stance |

In [7]:
class LinkedInOutput(BaseModel):
    hook: str
    body: str
    hashtags: List[str]
    format_used: str


writer = Agent(
    name="Writer",
    model="gpt-5-mini",
    instructions=(
        "You are a professional LinkedIn content writer. You receive a user query, research findings, "
        "and a professional analysis, plus a format type. Follow these format rules exactly:\n\n"

        "SHORT_POST (150-300 words):\n"
        "- hook: bold claim or question on the first line.\n"
        "- body: 3-5 short bullet insights, end with a 1-line CTA, 2-4 emojis placed naturally.\n"
        "- hashtags: 4-5 tags (broad reach + topic-specific).\n\n"

        "LONG_ARTICLE (700-1200 words):\n"
        "- hook: compelling 1-2 sentence editorial opening.\n"
        "- body: Intro → 3-4 insight sections → Takeaway, at least 2 factual references embedded, emojis at section breaks only.\n"
        "- hashtags: 5-7 tags including #ThoughtLeadership.\n\n"

        "CELEBRATION (100-200 words):\n"
        "- hook: milestone stated on the first line.\n"
        "- body: genuine emotion and gratitude, 3-5 celebratory emojis placed naturally.\n"
        "- hashtags: 3-4 tags relevant to the achievement domain.\n\n"

        "REACTION (150-250 words):\n"
        "- hook: clear stance opener — 'Hot take:', 'Counterpoint:', or 'This changes everything:'.\n"
        "- body: trend summary in 1-2 lines, 2-3 original insights, closing question to drive comments.\n"
        "- hashtags: 4-5 tags including #HotTake or similar.\n\n"

        "HASHTAG RULES (all formats):\n"
        "- At least 1 broad reach tag (#Innovation, #AI, #Leadership, #FutureOfWork).\n"
        "- At least 2 topic-specific tags from the query.\n"
        "- 1 format-relevant tag (#ThoughtLeadership, #Grateful, #HotTake).\n"
        "- Hashtags go ONLY in the hashtags field — never inside hook or body."
    ),
    output_type=LinkedInOutput,
)

In [8]:
async def manager_run(user_query: str, format_type: str) -> LinkedInOutput:
    import uuid

    valid_formats = {"short_post", "long_article", "celebration", "reaction"}
    if format_type not in valid_formats:
        raise ValueError(f"format_type must be one of {valid_formats}")

    session = SQLiteSession(str(uuid.uuid4()))

    # Step 1: Researcher gathers real-time facts
    research_result = await Runner.run(researcher, input=user_query, session=session)
    research_notes = research_result.final_output.summary

    # Step 2: Analyst extracts professional-audience insights
    analyst_result = await Runner.run(analyst, input=research_notes, session=session)
    analysis = analyst_result.final_output.summary

    # Step 3: Writer produces the LinkedIn post/article
    writer_input = (
        f"Format: {format_type.upper()}\n\n"
        f"User query: {user_query}\n\n"
        f"Research findings:\n{research_notes}\n\n"
        f"Analysis:\n{analysis}\n\n"
        f"Write a LinkedIn {format_type.replace('_', ' ')} on this topic. "
        f"Follow the {format_type.upper()} format rules exactly."
    )
    writer_result = await Runner.run(writer, input=writer_input, session=session)
    return writer_result.final_output

In [9]:
# --- Set your inputs here ---
user_query = "The rise of AI agents in enterprise software"
format_type = "short_post"   # short_post | long_article | celebration | reaction
# ----------------------------

post = await manager_run(user_query, format_type)

print("=== HOOK ===")
print(post.hook)

print("\n=== BODY ===")
print(post.body)

print("\n=== HASHTAGS ===")
print("  ".join(post.hashtags))

print(f"\n[Format: {post.format_used}]")

=== HOOK ===
AI agents will automate entire workflows — faster than most leaders expect.

=== BODY ===
- Market is already big and accelerating: ~$5.2B in 2025 with ~40%+ CAGR — agents are enterprise-grade now 🚀
- Vendors are racing (Microsoft, OpenAI, Google, Salesforce, ServiceNow); platform choice matters for lock‑in and governance 🧭
- Adoption is moving from pilots to production in CRM, customer service, HR and spreadsheets — prioritize high‑ROI domains 🔧
- Risk first: only ~1 in 5 orgs have mature agent governance; agent sprawl, ownership and observability are procurement must‑haves ⚠️
How are you preparing your stack and skills for agentic workflows?

=== HASHTAGS ===
#AI  #AIAgents  #EnterpriseSoftware  #Governance  #ThoughtLeadership

[Format: SHORT_POST]


## Export to Markdown

Call `export_to_md(post)` after any `manager_run` call to save the output as a formatted `.md` file in the project folder.

In [ ]:
from datetime import datetime
from pathlib import Path


def export_to_md(post: LinkedInOutput, query: str = "", output_dir: str = ".") -> str:
    """Save a LinkedInOutput as a formatted Markdown file. Returns the file path."""
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = post.format_used.replace(" ", "_").lower()
    filename = f"linkedin_{slug}_{timestamp}.md"
    output_dir="./outputs"  # Set your desired output directory here
    filepath = Path(output_dir) / filename

    hashtag_line = "  ".join(f"`{tag}`" for tag in post.hashtags)

    md = f"""# {post.hook}

{post.body}

---

**Hashtags:** {hashtag_line}

---
*Format: {post.format_used} · Generated: {datetime.now().strftime("%Y-%m-%d %H:%M")}*
"""
    if query:
        md = f"> **Query:** {query}\n\n" + md

    filepath.write_text(md, encoding="utf-8")
    print(f"✅ Saved: {filepath.resolve()}")
    return str(filepath.resolve())


# Export the last result
export_to_md(post, query=user_query)

FileNotFoundError: [Errno 2] No such file or directory: '\\outputs\\linkedin_short_post_20260930_143438.md'